## Example 1: Multipart file upload with MultipartEncoder

In [ ]:
# Example 1: Multipart file upload with MultipartEncoder
from requests_toolbelt.multipart.encoder import MultipartEncoder, MultipartEncoderMonitor
from requests_toolbelt.multipart.decoder import MultipartDecoder
import io

print("=== Multipart File Upload ===")

sample_files = {
    'document.txt': b'This is a sample text document for upload testing.',
    'data.csv': b'Name,Age,City\nAlice,28,New York\nBob,34,San Francisco',
    'image.bin': b'\x89PNG\r\n\x1a\n' + bytes(range(32)),
}

fields = {
    'title': 'Document Upload',
    'description': 'Multiple file upload example',
    'file1': ('document.txt', io.BytesIO(sample_files['document.txt']), 'text/plain'),
    'file2': ('data.csv', io.BytesIO(sample_files['data.csv']), 'text/csv'),
    'file3': ('image.bin', io.BytesIO(sample_files['image.bin']), 'application/octet-stream'),
}
encoder = MultipartEncoder(fields=fields)
content_type = encoder.content_type
expected_length = encoder.len
assert content_type.startswith('multipart/form-data; boundary='), content_type
assert content_type.endswith(encoder.boundary_value), (content_type, encoder.boundary_value)

# to_string() reads the whole body; it contains every field
body = encoder.to_string()
assert len(body) == expected_length, (len(body), expected_length)
for value in (b'Document Upload', b'Multiple file upload example', *sample_files.values()):
    assert value in body, value
for filename in sample_files:
    assert f'filename="{filename}"'.encode() in body, filename
print(f"Content-Type: {content_type}")
print(f"Body: {len(body)} bytes, {len(fields)} fields")

# Decoding the body gives back the same parts, in order
parts = MultipartDecoder(body, content_type).parts
assert [part.content for part in parts] == [b'Document Upload', b'Multiple file upload example', *sample_files.values()]
assert parts[2].headers[b'Content-Type'] == b'text/plain'

# MultipartEncoderMonitor calls back as the body is read
progress = []
monitor = MultipartEncoderMonitor(
    MultipartEncoder(fields={'blob': ('blob.bin', io.BytesIO(b'x' * 10000), 'application/octet-stream')}),
    lambda m: progress.append(m.bytes_read),
)
streamed = b''.join(iter(lambda: monitor.read(1024), b''))
assert len(streamed) == monitor.len and progress[-1] == monitor.len, (len(streamed), monitor.len, progress[-1:])
assert progress == sorted(progress) and len(progress) > 1, progress
print(f"Monitor: {len(progress)} callbacks, final bytes_read={progress[-1]}")

## Example 2: StreamingIterator for uploads

In [ ]:
# Example 2: StreamingIterator for uploads
from requests_toolbelt.streaming_iterator import StreamingIterator

print("Testing requests-toolbelt StreamingIterator:")

chunks = [
    b"Chunk 1: Sample data for streaming upload.\n",
    b"Chunk 2: More data being streamed.\n",
    b"Chunk 3: Final chunk of data.\n",
]
total_size = sum(len(chunk) for chunk in chunks)

# StreamingIterator wraps a generator in a file-like object that requests can upload
stream = StreamingIterator(total_size, (chunk for chunk in chunks))
assert stream.len == total_size

# Read it back in 4-byte pieces, the way an HTTP adapter would consume it
data = b''.join(iter(lambda: stream.read(4), b''))
assert data == b''.join(chunks), data
print(f"✓ Read {len(data)} bytes back from the iterator ({total_size} expected)")

## Example 3: Authentication helpers

In [ ]:
# Example 3: Authentication helpers
from requests_toolbelt.auth.guess import GuessAuth
from requests_toolbelt.auth.http_proxy_digest import HTTPProxyDigestAuth
from requests_toolbelt.auth.handler import AuthHandler
import requests

print("=== Authentication Extensions ===")

# GuessAuth answers a 401 challenge with Basic or Digest, so it hooks into the response
guess_auth = GuessAuth('api_user', 'api_password')
prepared = requests.Request('GET', 'https://example.org/', auth=guess_auth).prepare()
assert (guess_auth.username, guess_auth.password) == ('api_user', 'api_password')
assert len(prepared.hooks['response']) == 1, prepared.hooks
print(f"GuessAuth registered a response hook: {prepared.hooks['response'][0].__name__}")

# HTTPProxyDigestAuth is a Digest variant for proxies
proxy_auth = HTTPProxyDigestAuth('proxy_user', 'proxy_pass')
assert isinstance(proxy_auth, requests.auth.HTTPDigestAuth) and proxy_auth.username == 'proxy_user'

# AuthHandler picks credentials by host
handler = AuthHandler({'https://api.example.com': ('user', 'pass')})
strategy = handler.get_strategy_for('https://api.example.com/v1/items')
assert isinstance(strategy, requests.auth.HTTPBasicAuth) and strategy.username == 'user', strategy
other = handler.get_strategy_for('https://other.example.com/')
assert not isinstance(other, requests.auth.HTTPBasicAuth), other
basic = requests.Request('GET', 'https://api.example.com/v1/items', auth=handler).prepare()
assert basic.headers['Authorization'].startswith('Basic '), basic.headers
print("AuthHandler: Basic credentials for api.example.com only")

## Example 4: Multipart decoder and user agent

In [ ]:
# Example 4: Multipart decoder and user agent
from requests_toolbelt.multipart.decoder import MultipartDecoder
from requests_toolbelt import user_agent

print("Testing multipart decoder and utilities:")

boundary = "----Boundary123"
multipart_data = (
    b'------Boundary123\r\nContent-Disposition: form-data; name="field1"\r\n\r\nvalue1\r\n'
    b'------Boundary123\r\nContent-Disposition: form-data; name="file"; filename="test.txt"\r\n'
    b'Content-Type: text/plain\r\n\r\nFile content here\r\n'
    b'------Boundary123--\r\n'
)
decoder = MultipartDecoder(multipart_data, f"multipart/form-data; boundary={boundary}")
assert len(decoder.parts) == 2, decoder.parts
assert decoder.parts[0].text == 'value1'
assert decoder.parts[1].content == b'File content here'
assert decoder.parts[1].headers[b'Content-Type'] == b'text/plain'
for i, part in enumerate(decoder.parts, 1):
    print(f"✓ Part {i}: {len(part.content)} bytes")

# user_agent() builds "name/version extras... implementation/version system/release"
ua1 = user_agent('MyApp', '1.0.0')
ua2 = user_agent('TestClient', '2.5.3', extras=[('CustomLib', '1.2.0')])
assert ua1.startswith('MyApp/1.0.0 '), ua1
assert ua2.startswith('TestClient/2.5.3 CustomLib/1.2.0 '), ua2
print(f"✓ User agent 1: {ua1}")
print(f"✓ User agent 2: {ua2}")

## Example 5: Session adapters

In [ ]:
# Example 5: Session adapters
from requests_toolbelt.adapters.ssl import SSLAdapter
from requests_toolbelt.adapters.source import SourceAddressAdapter
import requests

print("Testing requests-toolbelt session adapters:")

session = requests.Session()

# SSLAdapter lets a session choose the TLS version for one URL prefix
ssl_adapter = SSLAdapter()
session.mount('https://example.org/', ssl_adapter)

# SourceAddressAdapter binds outgoing connections to a local address
source_adapter = SourceAddressAdapter(('127.0.0.1', 0))
session.mount('http://example.com/', source_adapter)

assert session.get_adapter('https://example.org/x') is ssl_adapter
assert session.get_adapter('http://example.com/api') is source_adapter
assert source_adapter.source_address == ('127.0.0.1', 0)
# Other URLs keep the default adapter
assert type(session.get_adapter('https://example.net/')) is requests.adapters.HTTPAdapter
for prefix, adapter in session.adapters.items():
    print(f"  {prefix} -> {type(adapter).__name__}")